In [1]:
import pandas as pd 
import numpy as np
import os
from pandas import DataFrame, Series
import plotly.graph_objects as go

In [118]:
def get_fe_price_data(
        filename: str = "FE_V2_GBPUSD_15mins_1yr_End_20250311.csv"
        ) -> DataFrame:
    """
    Return the FE price data as a DatetimeIndexed 
    DataFrame set to US/Eastern TZ
    """
    FOLDER = "price_data"
    PATH = f"{os.getcwd()}/{FOLDER}"
    df = pd.read_csv(f"{PATH}/{filename}")
    df["Date"] = pd.DatetimeIndex(df["Date"], tz="US/Eastern")
    df.set_index("Date", inplace=True)
    return df

In [119]:
fn = "FE_GBPUSD_15mins_1yr_End_20260311.csv"
data = get_fe_price_data(filename=fn)

/var/folders/zj/znqgk8kn6rqg7hbspq68r8_h0000gp/T/ipykernel_87870/1547086338.py:10: DtypeWarning: Columns (59,60,61,62,63,64,65,66,71,72,73,74,75,76,78,79,80,81,89,95,96,97) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{PATH}/{filename}")


In [430]:
base_cols = [
        "Iday_Idx", "Iday_Range","Yday_Range", "ADR", "Range", "ATR4", "Body", "RSI_DVG", "RSI", 
        "SMA4_Slope_SMA", "SMA16_Slope_SMA", "SMA32_Slope_SMA","Close_Pct_SMA", "Daily_IB", "IHR", "ILR"]

In [431]:
class TradingRange():
    """Define a daily trading range"""
    def __init__(self) -> None:
        self.range_high = None 
        self.range_low = None
        self.range_open = None 
        self.range_close = None
        self.range_hclose = None
        self.range_lclose = None

    def daily_inside_bar(
            self, 
            df: Series, 
            yday_high: Series, 
            yday_low: Series,
            yday_open: Series,
            yday_close: Series,
            yday_hclose: Series,
            yday_lclose: Series
            ):
        """Return index of daily trading range"""
        inside_day = False
        idx = int(df["Idx"])
        if df["Day_Idx"] > 0 and df["Iday_Idx"] == 0:
            yday_open_close_max = max([df["Yday_Close"],df["Yday_Open"]])
            yday_open_close_min = min([df["Yday_Close"],df["Yday_Open"]])
            if self.range_high is None and self.range_low is None:
                self.range_high = yday_high.iloc[idx-1]
                self.range_low = yday_low.iloc[idx-1]
                self.range_open = yday_open.iloc[idx-1]
                self.range_close = yday_close.iloc[idx-1]
                self.range_hclose = yday_hclose.iloc[idx-1]
                self.range_lclose = yday_lclose.iloc[idx-1]
            if self.range_high is not None and self.range_low is not None\
                and yday_open_close_max < self.range_high and yday_open_close_min > self.range_low:
                pass
            else:
                self.range_high = None 
                self.range_low = None
        if self.range_high is not None and self.range_low is not None:
            inside_day = True

        return inside_day

In [432]:
tr = TradingRange()
d_range_cols = ["Daily_IB", "MB_High","MB_Low","MB_Open","MB_Close","MB_HClose","MB_LClose"]
data["Daily_IB"] = data.apply(tr.daily_inside_bar, axis=1, args=[data["Yday_High"], data["Yday_Low"]])


In [402]:
data[base_cols].at_time("17:15").query("Daily_IB == True")

,Iday_Idx,Iday_Range,Yday_Range,ADR,Range,ATR4,Body,RSI_DVG,RSI,SMA4_Slope_SMA,SMA16_Slope_SMA,SMA32_Slope_SMA,Close_Pct_SMA,Daily_IB
Date,,,,,,,,,,,,,,
2025-03-13 17:15:00-04:00,0,0.000405,0.005250,NaN,0.000405,0.000524,0.000155,NaN,53.699461,17.557484,6.961392,-0.373162,0.033573,True
2025-03-16 17:15:00-04:00,0,0.000725,0.004795,NaN,0.000725,0.000504,0.000205,NaN,48.356022,4.474393,5.953221,-6.265584,0.009497,True
2025-03-19 17:15:00-04:00,0,0.000880,0.005595,NaN,0.000880,0.000570,0.000320,NaN,59.331009,45.025605,37.014209,20.439340,0.074067,True
2025-03-20 17:15:00-04:00,0,0.000965,0.007875,NaN,0.000965,0.000486,0.000355,NaN,55.472742,20.188959,4.182293,11.136652,0.042548,True
2025-03-24 17:15:00-04:00,0,0.000320,0.008090,NaN,0.000320,0.000349,0.000290,NaN,48.085950,8.422697,-4.475309,-35.418566,0.010692,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-04 17:15:00-05:00,0,0.000340,0.010000,0.010048,0.000340,0.000484,0.000035,NaN,54.364148,-20.130700,5.708960,-8.669633,0.043949,True
2026-03-05 17:15:00-05:00,0,0.001250,0.008960,0.009633,0.001250,0.000700,0.001070,NaN,58.559745,12.354379,25.152410,-11.836124,0.089900,True
2026-03-08 17:15:00-04:00,0,0.000545,0.010475,0.009752,0.000545,0.001059,0.000280,NaN,38.809031,28.018922,25.110044,47.832370,0.304245,True


In [436]:
data[base_cols].query("Daily_IB == True and (IHR == True or ILR == True)").tail(51)

,Iday_Idx,Iday_Range,Yday_Range,ADR,Range,ATR4,Body,RSI_DVG,RSI,SMA4_Slope_SMA,SMA16_Slope_SMA,SMA32_Slope_SMA,Close_Pct_SMA,Daily_IB,IHR,ILR
Date,,,,,,,,,,,,,,,,
2026-01-28 23:00:00-05:00,23,0.004275,0.010145,0.009370,0.000710,0.000796,0.000285,NaN,62.892973,67.811826,18.811785,37.481527,0.100788,True,True,NaN
2026-01-28 23:30:00-05:00,25,0.004295,0.010145,0.009370,0.000625,0.000835,0.000040,NaN,63.150414,62.555969,13.312548,36.921835,0.102547,True,True,NaN
2026-01-30 04:45:00-05:00,46,0.008955,0.010570,0.009640,0.001690,0.001391,0.000680,True,34.938121,-73.306389,3.889972,-37.643247,0.159298,True,NaN,True
2026-01-30 10:00:00-05:00,67,0.009500,0.010570,0.009640,0.002410,0.001700,0.001295,True,35.394083,-48.431186,11.987478,6.051567,0.231066,True,NaN,True
2026-01-30 10:30:00-05:00,69,0.009845,0.010570,0.009640,0.002360,0.001991,0.000425,True,41.328807,-58.855223,3.551020,1.548948,0.133752,True,NaN,True
2026-01-30 16:00:00-05:00,91,0.013555,0.010570,0.009640,0.000725,0.000814,0.000215,True,36.713653,-67.372913,-55.423783,-48.314981,0.098890,True,NaN,True
2026-02-02 00:45:00-05:00,30,0.004540,0.013555,0.010400,0.001190,0.000915,0.000350,NaN,40.439890,-45.738417,-6.194653,-14.659512,0.076625,True,NaN,True
2026-02-03 11:00:00-05:00,71,0.005605,0.009220,0.010476,0.000915,0.001274,0.000125,NaN,61.170092,75.701207,12.228462,-15.248416,0.158188,True,True,NaN
2026-02-03 12:15:00-05:00,76,0.005650,0.009220,0.010476,0.001120,0.000979,0.000330,NaN,58.725145,-4.195921,31.982807,-5.493139,0.093974,True,True,NaN
